# FRF Dataset Builder

Loads 13 `.npy` FRF files, filters to a frequency range, and builds the final dataset for PCA or downstream processing.

**Shape at each stage:**
- Each file loaded: `(60, 9, 4099)` — 60 trials, 9 points, full frequency axis
- After frequency filtering: `(60, 9, 381)` — trimmed to 10–200 Hz
- After stacking all 13 classes: `(780, 9, 381)`

---
## Cell 1 — Imports & Configuration

Set your folder path, frequency bounds, and class order here.

In [ ]:
import glob
import numpy as np
import os
import json
from sklearn.decomposition import PCA

# ── Folder containing your .npy files ────────────────────────────────────────
path_folder = r"../450,60,2 Dataset/processed_dataset3"

# ── Frequency axis settings ───────────────────────────────────────────────────
freq_resolution = 0.5   # Hz per index step (your data runs 0 → 2048.5 Hz)
freq_start      = 10    # Hz — lower bound of the band you want to keep
freq_end        = 200   # Hz — upper bound

# Convert Hz → array indices
# e.g. 10 Hz / 0.5 Hz_per_step = index 20
idx_start = int(freq_start / freq_resolution)
idx_end   = int(freq_end   / freq_resolution) + 1   # +1 so slice includes freq_end

print(f"Frequency band  : {freq_start} Hz  →  {freq_end} Hz")
print(f"Index range     : {idx_start}  →  {idx_end - 1}")
print(f"Points kept     : {idx_end - idx_start}")

# ── Class order (must match your file naming convention) ─────────────────────
class_order = [
    'UD',
    'LD_E1', 'LD_E2', 'LD_E3', 'LD_E4',
    'MD_E1', 'MD_E2', 'MD_E3', 'MD_E4',
    'HD_E1', 'HD_E2', 'HD_E3', 'HD_E4',
]
print(f"\nClasses ({len(class_order)})      : {class_order}")

output_dir = "PCA_Data_HD"
os.makedirs(output_dir, exist_ok=True)

output_X       = os.path.join(output_dir, "PCA_Frf.npy")
output_y       = os.path.join(output_dir, "y_train.npy")
output_frf     = os.path.join(output_dir, "frf_data.npy")
output_mapping = os.path.join(output_dir, "class_mapping.json")

Frequency band  : 10 Hz  →  200 Hz
Index range     : 20  →  400
Points kept     : 381

Classes (13)      : ['UD', 'LD_E1', 'LD_E2', 'LD_E3', 'LD_E4', 'MD_E1', 'MD_E2', 'MD_E3', 'MD_E4', 'HD_E1', 'HD_E2', 'HD_E3', 'HD_E4']


---
## Cell 2 — Discover Files

Scans `path_folder` for `.npy` files and maps them to class names.

In [ ]:
# Scan folder and strip the 'frf_' prefix and '_Group_final' suffix
# e.g.  frf_HD_E1_Group_final.npy  →  'HD_E1'
all_files_dict = {}
for file_path in glob.glob(os.path.join(path_folder, "*.npy")):
    file_name  = os.path.basename(file_path).replace(".npy", "")
    class_name = file_name.replace("frf_", "").replace("_Group_final", "")
    all_files_dict[class_name] = file_path
    print(f"  Found  {file_name}  →  class '{class_name}'")

# Build ordered list — classes not found are skipped with a warning
ordered_files = []
for cn in class_order:
    if cn in all_files_dict:
        ordered_files.append((cn, all_files_dict[cn]))
    else:
        print(f"  [WARNING] '{cn}' not found in folder — will be missing from dataset")

print(f"\n{len(ordered_files)} / {len(class_order)} classes ready")

  Found  frf_HD_E1_Group_final  →  class 'HD_E1'
  Found  frf_HD_E2_Group_final  →  class 'HD_E2'
  Found  frf_HD_E3_Group_final  →  class 'HD_E3'
  Found  frf_HD_E4_Group_final  →  class 'HD_E4'
  Found  frf_LD_E1_Group_final  →  class 'LD_E1'
  Found  frf_LD_E2_Group_final  →  class 'LD_E2'
  Found  frf_LD_E3_Group_final  →  class 'LD_E3'
  Found  frf_LD_E4_Group_final  →  class 'LD_E4'
  Found  frf_MD_E1_Group_final  →  class 'MD_E1'
  Found  frf_MD_E2_Group_final  →  class 'MD_E2'
  Found  frf_MD_E3_Group_final  →  class 'MD_E3'
  Found  frf_MD_E4_Group_final  →  class 'MD_E4'
  Found  frf_UD_Group_final  →  class 'UD'

13 / 13 classes ready


---
## Cell 3 — (Optional) Replace One or More Cases

Use this cell if you want to swap out a class file with a different `.npy` file —
for example, replacing `HD_E1` with your newly processed file.

Set `replacements = {}` (empty dict) to skip this step.

In [ ]:
# Edit this dict to replace any class with a custom file path
# Format:  'CLASS_NAME' : r'path/to/your/new_file.npy'
# Your replacement file should already be filtered to the correct frequency band (60, 9, 381)
# Leave as {} to use all original files unchanged

replacements = {
    'HD_E1' : r'D:\UM Document\FYP_ML\FEA_Data\FEA_Data_Merged\HD_E1_frf_all.npy',
    'HD_E2' : r'D:\UM Document\FYP_ML\FEA_Data\FEA_Data_Merged\HD_E2_frf_all.npy',
    'HD_E3' : r'D:\UM Document\FYP_ML\FEA_Data\FEA_Data_Merged\HD_E3_frf_all.npy',
    'HD_E4' : r'D:\UM Document\FYP_ML\FEA_Data\FEA_Data_Merged\HD_E4_frf_all.npy',
}

# replaced_classes is a set of class names that have been swapped out
# Cell 4 reads this to know which files to skip frequency slicing for
replaced_classes = set()

if replacements:
    updated_files = []
    for cn, fp in ordered_files:
        if cn in replacements:
            new_fp = replacements[cn]
            if os.path.isfile(new_fp):
                print(f"  Replacing '{cn}'")
                print(f"    old : {fp}")
                print(f"    new : {new_fp}  <- already filtered, slicing will be skipped")
                updated_files.append((cn, new_fp))
                replaced_classes.add(cn)
            else:
                print(f"  [ERROR] Replacement not found for '{cn}': {new_fp}")
                print(f"          Keeping original: {fp}")
                updated_files.append((cn, fp))
        else:
            updated_files.append((cn, fp))
    ordered_files = updated_files
    print(f"\nReplacements applied : {list(replaced_classes)}")
else:
    print("No replacements — using original files for all classes.")

print("\nFinal file list:")
for cn, fp in ordered_files:
    tag = '  <- pre-filtered (no slice)' if cn in replaced_classes else ''
    print(f"  {cn:10s}  {os.path.basename(fp)}{tag}")


  Replacing 'HD_E1'
    old : ../450,60,2 Dataset/processed_dataset3\frf_HD_E1_Group_final.npy
    new : D:\UM Document\FYP_ML\FEA_Data\FEA_Data_Merged\HD_E1_frf_all.npy  <- already filtered, slicing will be skipped
  Replacing 'HD_E2'
    old : ../450,60,2 Dataset/processed_dataset3\frf_HD_E2_Group_final.npy
    new : D:\UM Document\FYP_ML\FEA_Data\FEA_Data_Merged\HD_E2_frf_all.npy  <- already filtered, slicing will be skipped
  Replacing 'HD_E3'
    old : ../450,60,2 Dataset/processed_dataset3\frf_HD_E3_Group_final.npy
    new : D:\UM Document\FYP_ML\FEA_Data\FEA_Data_Merged\HD_E3_frf_all.npy  <- already filtered, slicing will be skipped
  Replacing 'HD_E4'
    old : ../450,60,2 Dataset/processed_dataset3\frf_HD_E4_Group_final.npy
    new : D:\UM Document\FYP_ML\FEA_Data\FEA_Data_Merged\HD_E4_frf_all.npy  <- already filtered, slicing will be skipped

Replacements applied : ['HD_E1', 'HD_E2', 'HD_E3', 'HD_E4']

Final file list:
  UD          frf_UD_Group_final.npy
  LD_E1       frf_LD

---
## Cell 4 — Load & Filter to Frequency Band

Each file is loaded as `(60, 9, full_freq)` and immediately sliced to `(60, 9, 381)` using `idx_start:idx_end`.

The full array is kept complex — magnitude conversion happens later (in PCA).

In [ ]:
frf_list     = []   # will hold one (60, 9, 381) array per class
class_labels = []   # class name matching each entry in frf_list

for class_idx, (cn, fp) in enumerate(ordered_files):
    data = np.load(fp)

    if cn in replaced_classes:
        # Replacement file is already filtered to the correct frequency band
        # use it as-is, no slicing needed
        data_filtered = data
        print(f"  {cn:10s}  loaded {data.shape}  (pre-filtered, no slice applied)")
    else:
        # Original file covers the full frequency axis
        # slice axis 2 to keep only idx_start:idx_end  (freq_start -> freq_end Hz)
        data_filtered = data[:, :, idx_start:idx_end]
        print(f"  {cn:10s}  loaded {data.shape}  ->  sliced to {data_filtered.shape}")

    frf_list.append(data_filtered)
    class_labels.append(cn)

print(f"\nAll {len(frf_list)} classes loaded.")

  UD          loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  LD_E1       loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  LD_E2       loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  LD_E3       loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  LD_E4       loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  MD_E1       loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  MD_E2       loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  MD_E3       loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  MD_E4       loaded (60, 9, 2049)  ->  sliced to (60, 9, 381)
  HD_E1       loaded (60, 9, 381)  (pre-filtered, no slice applied)
  HD_E2       loaded (60, 9, 381)  (pre-filtered, no slice applied)
  HD_E3       loaded (60, 9, 381)  (pre-filtered, no slice applied)
  HD_E4       loaded (60, 9, 381)  (pre-filtered, no slice applied)

All 13 classes loaded.


---
## Cell 5 — Stack into Final 3D Array

`np.concatenate` along axis 0 stacks the 13 × `(60, 9, 381)` blocks into `(780, 9, 381)`.

In [ ]:
# Stack along axis 0 (the trials/samples axis)
frf_data = np.concatenate(frf_list, axis=0)   # (780, 9, 381)

print(f"frf_data shape : {frf_data.shape}")
print(f"  axis 0 = trials  ({frf_data.shape[0]} total = {len(class_order)} classes × 60 trials)")
print(f"  axis 1 = points  ({frf_data.shape[1]} → pt1 to pt9)")
print(f"  axis 2 = freq    ({frf_data.shape[2]} points covering {freq_start}–{freq_end} Hz)")
print(f"dtype          : {frf_data.dtype}")

# Build matching label array (y)
# e.g. first 60 rows → class 0 (UD), next 60 → class 1 (LD_E1), ...
y = np.repeat(np.arange(len(ordered_files)), 60)
class_mapping = {i: cn for i, (cn, _) in enumerate(ordered_files)}

print(f"\ny shape        : {y.shape}")
print("Class mapping  :")
for idx, cn in class_mapping.items():
    print(f"  {idx:2d} → {cn}")

frf_data shape : (780, 9, 381)
  axis 0 = trials  (780 total = 13 classes × 60 trials)
  axis 1 = points  (9 → pt1 to pt9)
  axis 2 = freq    (381 points covering 10–200 Hz)
dtype          : complex128

y shape        : (780,)
Class mapping  :
   0 → UD
   1 → LD_E1
   2 → LD_E2
   3 → LD_E3
   4 → LD_E4
   5 → MD_E1
   6 → MD_E2
   7 → MD_E3
   8 → MD_E4
   9 → HD_E1
  10 → HD_E2
  11 → HD_E3
  12 → HD_E4


---
## Cell 6 — PCA Feature Extraction

For each trial and each frequency band, we:
1. Take the `(9, 381)` slice (9 points × 381 frequencies)
2. Transpose to `(381, 9)` — PCA expects samples × features
3. Take the magnitude (complex → real)
4. Project onto the first principal component → `(381,)` vector

In [ ]:
pca    = PCA(n_components=1)
X_list = []

for trial_idx in range(frf_data.shape[0]):   # loop over 780 trials
    # frf_data[trial_idx] → shape (9, 381)
    # .T                  → shape (381, 9)  ← PCA wants (samples, features)
    trial = np.abs(frf_data[trial_idx].T)     # magnitude: (381, 9)

    pca.fit(trial)
    # project: (381, 9) @ (9, 1) → (381, 1) → flatten → (381,)
    signal = trial.dot(pca.components_.T).flatten()

    X_list.append(signal)

X = np.array(X_list)   # (780, 381)

print(f"X shape : {X.shape}  — (trials, freq_points)")
print(f"y shape : {y.shape}  — class label per trial")

X shape : (780, 381)  — (trials, freq_points)
y shape : (780,)  — class label per trial


---
## Cell 7 — Save Outputs

In [ ]:
# frf_data saved as magnitude (real) for downstream use
np.save(output_X,   X)
np.save(output_y,   y)
np.save(output_frf, np.abs(frf_data))   # complex → magnitude before saving

with open(output_mapping, "w") as f:
    json.dump(class_mapping, f, indent=4)

print("Saved:")
print(f"  PCA features   : {output_X}       shape {X.shape}")
print(f"  Labels         : {output_y}    shape {y.shape}")
print(f"  Raw FRF (|·|)  : {output_frf}  shape {np.abs(frf_data).shape}")
print(f"  Class mapping  : {output_mapping}")

Saved:
  PCA features   : PCA_Data_HD\PCA_Frf.npy       shape (780, 381)
  Labels         : PCA_Data_HD\y_train.npy    shape (780,)
  Raw FRF (|·|)  : PCA_Data_HD\frf_data.npy  shape (780, 9, 381)
  Class mapping  : PCA_Data_HD\class_mapping.json


---
## Cell 8 — Verify

Reload everything and confirm shapes and class distribution.

In [ ]:
check_X   = np.load(output_X)
check_y   = np.load(output_y)
check_frf = np.load(output_frf)

print(f"X shape       : {check_X.shape}")
print(f"y shape       : {check_y.shape}")
print(f"frf_data shape: {check_frf.shape}")
print(f"\nClass distribution:")
for idx, cn in class_mapping.items():
    count = np.sum(check_y == idx)
    print(f"  {idx:2d}  {cn:10s}  {count} trials")

print(f"\nFrequency axis  : {freq_start} Hz → {freq_end} Hz  ({check_frf.shape[2]} points, step={freq_resolution} Hz)")
# Reconstruct the actual Hz values for reference
freq_axis = np.arange(check_frf.shape[2]) * freq_resolution + freq_start
print(f"First 5 freq values : {freq_axis[:5]}")
print(f"Last  5 freq values : {freq_axis[-5:]}")

X shape       : (780, 381)
y shape       : (780,)
frf_data shape: (780, 9, 381)

Class distribution:
   0  UD          60 trials
   1  LD_E1       60 trials
   2  LD_E2       60 trials
   3  LD_E3       60 trials
   4  LD_E4       60 trials
   5  MD_E1       60 trials
   6  MD_E2       60 trials
   7  MD_E3       60 trials
   8  MD_E4       60 trials
   9  HD_E1       60 trials
  10  HD_E2       60 trials
  11  HD_E3       60 trials
  12  HD_E4       60 trials

Frequency axis  : 10 Hz → 200 Hz  (381 points, step=0.5 Hz)
First 5 freq values : [10.  10.5 11.  11.5 12. ]
Last  5 freq values : [198.  198.5 199.  199.5 200. ]
